# Training record of the corrected_v1 models (original notebook 39, cleaned)

This is the **original record** of the run that trained the ParticleVision models (`notebooks/39_corrected_data_retrain_and_reevaluate.ipynb`, run 29–30 Aug 2026 on a Mac with the PyTorch MPS backend, Python 3.9.6), **cleaned of images and paths**:

- image outputs removed (3): they show lab data;
- absolute and personal paths replaced with `<project>`, `<home>`, `<site-packages>`, `<python>`;
- machine-specific environment warnings removed (4 lines).

The code and the text outputs (losses, metrics, epochs, timings) are otherwise unchanged. It is a record, not meant to be re-run from here: to train, use `training/train_unet.py` and `training/train_detector.py` (see `training/README.md`). Cells 17–24 are the evaluation of the trained models.

# Notebook 39 — Retrain on Corrected Data & Re-Evaluate

This notebook retrains the **confirmed Gated Fusion recipe** (Branch 1 U-Net + Branch 2
FinesDetector, gated continuous-prior fusion, `fusion_scale=8.0`, `fusion_margin=2.0`, no TTA)
on the **corrected, de-duplicated dataset**, then runs the full official evaluation on the
corrected 16-image test set.

**What changed vs. the original confirmed run:** only the data. `data_split.json` was updated
and two label files were spatially corrected — see `CORRECTIONS_LOG.md` for the full list and
reasons. The model architectures, losses, optimizers, schedules, and fusion parameters below are
copied verbatim from the notebooks that produced the original confirmed checkpoints
(`notebooks/02_training.ipynb` for Branch 1, `notebooks/28_twin_branch_fines.ipynb` for Branch 2),
so this is a like-for-like "same recipe, corrected data" comparison.

**Checkpoints are written to a separate directory** (`checkpoints/corrected_v1/`) so the original
confirmed checkpoints (`stage3_unet_best.pth`, `fines_detector_best.pth`) are never touched and
remain available for reference/fallback.

Per project history of MPS memory instability on longer runs, Branch 1 now saves a checkpoint
**every epoch** (`{stage}_e{NN}.pth`, weights-only) in addition to its existing best/last
checkpoints — Branch 2 already saved every epoch natively in the original recipe.

**Run this notebook manually, top to bottom.** Branch 1's Stage 3 (100% of training data, 20
epochs) and Branch 2 (30 epochs) are the expensive steps — expect this to take hours depending on
hardware. If it crashes partway through, the per-epoch checkpoints let you resume without losing
all progress — just reload the latest `_eNN.pth` and continue rather than re-running from scratch.


In [1]:
import sys, os, time, json, gc
sys.path.append(os.path.abspath(".."))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import label, find_objects

from src.config import CFG
from src.model import build_unet, FinesDetector
from src.training import run_training_stage
from src.losses import ComboLoss
from src.inference import predict_full_image, normalize_image
from src.tiling import compute_tile_positions
from src.dataset import (
    SandPatchDataset, create_datasets,
    get_train_augmentation, get_val_augmentation, rgb_to_class_index,
)
from src.metrics import SegmentationMetrics
from src.evaluation import run_official_evaluation

CORRECTED_CKPT_DIR = os.path.join(CFG.project_root, "checkpoints", "corrected_v1")
os.makedirs(CORRECTED_CKPT_DIR, exist_ok=True)
print("Checkpoints for this run will be written to:", CORRECTED_CKPT_DIR)
print("Original confirmed checkpoints are NOT touched.")


Checkpoints for this run will be written to: <project>/checkpoints/corrected_v1
Original confirmed checkpoints are NOT touched.


## 0. Sanity check — confirm the corrected data is actually in place

This must show train=79, val=16, test=16 before anything else. If it doesn't, stop and re-check `data_split.json` before spending hours training on the wrong data.

In [2]:
with open(os.path.join(CFG.project_root, "data_split.json")) as f:
    split = json.load(f)

train_indices = split["train"]
val_indices = split["val"]
test_indices = split["test"]

print(f"train: {len(train_indices)} images")
print(f"val:   {len(val_indices)} images")
print(f"test:  {len(test_indices)} images")

assert len(train_indices) == 79, f"Expected 79 train images, got {len(train_indices)}"
assert len(val_indices) == 16, f"Expected 16 val images, got {len(val_indices)}"
assert len(test_indices) == 16, f"Expected 16 test images, got {len(test_indices)}"
for bad in [20, 29, 79, 99, 100]:
    assert bad not in train_indices, f"Image_{bad:03d} should have been removed from train"
assert 109 not in val_indices, "Image_109 should have been removed from val"
for bad in [90, 119]:
    assert bad not in test_indices, f"Image_{bad:03d} should have been removed from test"
assert 9 in train_indices and 19 in test_indices, "Image_009/019 should still be present (corrected, not excluded)"
print("\n\u2713 Split matches the corrected dataset exactly.")

# Confirm the corrected label files are actually in place (differ from the pre-correction backups)
backup_dir = os.path.join(CFG.raw_dir, "..", "Segmented_ORIGINAL_BACKUP")
for idx in [9, 19]:
    cur = np.array(Image.open(CFG.seg_path(idx)))
    bak_path = os.path.join(backup_dir, f"Image_{idx:03d}.tif")
    if os.path.exists(bak_path):
        bak = np.array(Image.open(bak_path))
        same_shape = cur.shape == bak.shape
        differs = not np.array_equal(cur, bak)
        print(f"Image_{idx:03d}: shape matches backup={same_shape}, content differs from original={differs}")
        assert differs, f"Image_{idx:03d} label file does not differ from its pre-correction backup!"
    else:
        print(f"Image_{idx:03d}: no backup found at {bak_path} (skipping content check)")
print("\n\u2713 Corrected label files confirmed in place.")


train: 79 images
val:   16 images
test:  16 images

✓ Split matches the corrected dataset exactly.
Image_009: shape matches backup=True, content differs from original=True
Image_019: shape matches backup=True, content differs from original=True

✓ Corrected label files confirmed in place.


## 1. Branch 1 — U-Net (EfficientNet-B3), 3-stage progressive training

Identical recipe to `notebooks/02_training.ipynb`: `ComboLoss`, `CFG.training_stages =
[(0.20, 30), (0.60, 20), (1.00, 20)]`, `run_training_stage` with default optimizer/scheduler.
The only difference from the original run: `checkpoint_dir` points at `corrected_v1/`, stage
names get a `_corrected` suffix, and `save_every_epoch=True` is now passed through.

In [3]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {device}")

model = build_unet().to(device)
criterion = ComboLoss().to(device)
print(f"Loss: ComboLoss (Focal weight={CFG.loss_focal_weight}, Dice weight={CFG.loss_dice_weight})")
print(f"  Focal alpha per class: {CFG.focal_alpha}")
print(f"  Focal gamma: {CFG.focal_gamma}")
print(f"  Dice fines weight: {CFG.dice_fines_weight}")


Device: mps
Loss: ComboLoss (Focal weight=0.5, Dice weight=0.5)
  Focal alpha per class: [1.0, 1.8, 2.9, 50.0]
  Focal gamma: 2.0
  Dice fines weight: 3.0


In [4]:
# ── Stage 1: 20% of training data ──────────────────────────────────
stage1_fraction = CFG.training_stages[0][0]
stage1_epochs   = CFG.training_stages[0][1]
print(f"Stage 1: {stage1_fraction*100:.0f}% of training data, {stage1_epochs} epochs")

t0 = time.time()
train_ds_s1, val_ds, test_ds = create_datasets(train_fraction=stage1_fraction, seed=CFG.seed)
print(f"Dataset creation took {time.time()-t0:.1f}s")

train_loader_s1 = DataLoader(train_ds_s1, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=CFG.num_workers, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                         num_workers=CFG.num_workers, pin_memory=False)

print(f"Train: {len(train_ds_s1)} patches -> {len(train_loader_s1)} batches")
print(f"Val:   {len(val_ds)} patches -> {len(val_loader)} batches")


Stage 1: 20% of training data, 30 epochs
Creating datasets: train=15 imgs, val=16 imgs, test=16 imgs
  Train patches: 1198
  Val patches:   1260
  Test patches:  1248
Dataset creation took 23.2s
Train: 1198 patches -> 150 batches
Val:   1260 patches -> 158 batches


In [5]:
history_s1 = run_training_stage(
    model=model,
    train_loader=train_loader_s1,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    stage_name='stage1_unet_corrected',
    num_epochs=stage1_epochs,
    checkpoint_dir=CORRECTED_CKPT_DIR,
    save_every_epoch=True,
)



  Stage: stage1_unet_corrected
  Epochs: 30, LR: 0.0001, Patience: 7
  Train batches: 150, Val batches: 158
  [MPS Mem] Allocated: 51MB, Driver: 96MB


  Epoch   1/30 │ Loss: 0.5634/0.3956 │ mIoU: 0.6997 │ Fines IoU: 0.0688 │ LR: 9.76e-05 │ 284.8s
  [MPS Mem] Allocated: 319MB, Driver: 9434MB


  Epoch   2/30 │ Loss: 0.3612/0.3330 │ mIoU: 0.7517 │ Fines IoU: 0.1607 │ LR: 9.05e-05 │ 283.1s
  [MPS Mem] Allocated: 323MB, Driver: 9454MB


  Epoch   3/30 │ Loss: 0.3198/0.3089 │ mIoU: 0.7650 │ Fines IoU: 0.1944 │ LR: 7.94e-05 │ 280.8s
  [MPS Mem] Allocated: 324MB, Driver: 8430MB


  Epoch   4/30 │ Loss: 0.3021/0.2961 │ mIoU: 0.7706 │ Fines IoU: 0.2034 │ LR: 6.55e-05 │ 278.3s
  [MPS Mem] Allocated: 325MB, Driver: 8430MB


  Epoch   5/30 │ Loss: 0.2921/0.2840 │ mIoU: 0.7723 │ Fines IoU: 0.2064 │ LR: 5.00e-05 │ 278.3s
  [MPS Mem] Allocated: 324MB, Driver: 8430MB


... (12 outputs left out)

  Epoch  18/30 │ Loss: 0.1466/0.1569 │ mIoU: 0.8463 │ Fines IoU: 0.4776 │ LR: 6.55e-05 │ 276.2s
  [MPS Mem] Allocated: 325MB, Driver: 10474MB


  Epoch  19/30 │ Loss: 0.1411/0.1542 │ mIoU: 0.8450 │ Fines IoU: 0.4724 │ LR: 5.78e-05 │ 277.3s
  [MPS Mem] Allocated: 325MB, Driver: 10474MB


  Epoch  20/30 │ Loss: 0.1407/0.1554 │ mIoU: 0.8454 │ Fines IoU: 0.4752 │ LR: 5.00e-05 │ 276.7s
  [MPS Mem] Allocated: 324MB, Driver: 9458MB


  Epoch  21/30 │ Loss: 0.1376/0.1530 │ mIoU: 0.8436 │ Fines IoU: 0.4657 │ LR: 4.22e-05 │ 276.5s
  ⏹ Early stopping at epoch 21 (no improvement for 7 epochs)
  ✅ Restored best model from epoch 14 (val mIoU = 0.8468)


In [6]:
# ── Stage 2: 60% of training data ──────────────────────────────────
stage2_fraction = CFG.training_stages[1][0]
stage2_epochs   = CFG.training_stages[1][1]
print(f"Stage 2: {stage2_fraction*100:.0f}% of training data, {stage2_epochs} epochs")

del train_ds_s1, train_loader_s1
gc.collect()

t0 = time.time()
train_ds_s2, _, _ = create_datasets(train_fraction=stage2_fraction, seed=CFG.seed)
print(f"Dataset creation took {time.time()-t0:.1f}s")

train_loader_s2 = DataLoader(train_ds_s2, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=CFG.num_workers, pin_memory=False)
print(f"Train: {len(train_ds_s2)} patches -> {len(train_loader_s2)} batches")


Stage 2: 60% of training data, 20 epochs
Creating datasets: train=47 imgs, val=16 imgs, test=16 imgs
  Train patches: 3666
  Val patches:   1260
  Test patches:  1248
Dataset creation took 40.7s
Train: 3666 patches -> 459 batches


In [7]:
history_s2 = run_training_stage(
    model=model,
    train_loader=train_loader_s2,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    stage_name='stage2_unet_corrected',
    num_epochs=stage2_epochs,
    checkpoint_dir=CORRECTED_CKPT_DIR,
    save_every_epoch=True,
)



  Stage: stage2_unet_corrected
  Epochs: 20, LR: 0.0001, Patience: 7
  Train batches: 459, Val batches: 158
  [MPS Mem] Allocated: 139MB, Driver: 10464MB


  Epoch   1/20 │ Loss: 0.1922/0.1588 │ mIoU: 0.8414 │ Fines IoU: 0.4634 │ LR: 9.76e-05 │ 718.5s
  [MPS Mem] Allocated: 360MB, Driver: 11486MB


  Epoch   2/20 │ Loss: 0.1822/0.1526 │ mIoU: 0.8461 │ Fines IoU: 0.4791 │ LR: 9.05e-05 │ 713.9s
  [MPS Mem] Allocated: 359MB, Driver: 10474MB


  Epoch   3/20 │ Loss: 0.1790/0.1535 │ mIoU: 0.8404 │ Fines IoU: 0.4554 │ LR: 7.94e-05 │ 713.8s
  [MPS Mem] Allocated: 360MB, Driver: 10454MB


  Epoch   4/20 │ Loss: 0.1730/0.1482 │ mIoU: 0.8448 │ Fines IoU: 0.4757 │ LR: 6.55e-05 │ 715.1s
  [MPS Mem] Allocated: 360MB, Driver: 10454MB


  Epoch   5/20 │ Loss: 0.1692/0.1448 │ mIoU: 0.8490 │ Fines IoU: 0.4866 │ LR: 5.00e-05 │ 714.6s
  [MPS Mem] Allocated: 360MB, Driver: 11478MB


... (5 outputs left out)

  Epoch  11/20 │ Loss: 0.1644/0.1432 │ mIoU: 0.8467 │ Fines IoU: 0.4748 │ LR: 9.94e-05 │ 716.0s
  [MPS Mem] Allocated: 361MB, Driver: 11486MB


  Epoch  12/20 │ Loss: 0.1637/0.1390 │ mIoU: 0.8519 │ Fines IoU: 0.4978 │ LR: 9.76e-05 │ 712.5s
  [MPS Mem] Allocated: 361MB, Driver: 11486MB


  Epoch  13/20 │ Loss: 0.1616/0.1425 │ mIoU: 0.8456 │ Fines IoU: 0.4746 │ LR: 9.46e-05 │ 713.5s
  [MPS Mem] Allocated: 361MB, Driver: 11486MB


  Epoch  14/20 │ Loss: 0.1590/0.1391 │ mIoU: 0.8493 │ Fines IoU: 0.4889 │ LR: 9.05e-05 │ 713.0s
  ⏹ Early stopping at epoch 14 (no improvement for 7 epochs)
  ✅ Restored best model from epoch 7 (val mIoU = 0.8530)


In [8]:
# ── Stage 3: 100% of training data (this is the expensive one) ──────
stage3_fraction = CFG.training_stages[2][0]
stage3_epochs   = CFG.training_stages[2][1]
print(f"Stage 3: {stage3_fraction*100:.0f}% of training data, {stage3_epochs} epochs")

del train_ds_s2, train_loader_s2
gc.collect()

t0 = time.time()
train_ds_s3, _, _ = create_datasets(train_fraction=stage3_fraction, seed=CFG.seed)
print(f"Dataset creation took {time.time()-t0:.1f}s")

train_loader_s3 = DataLoader(train_ds_s3, batch_size=CFG.batch_size, shuffle=True,
                              num_workers=CFG.num_workers, pin_memory=False)
print(f"Train: {len(train_ds_s3)} patches -> {len(train_loader_s3)} batches")


Stage 3: 100% of training data, 20 epochs
Creating datasets: train=79 imgs, val=16 imgs, test=16 imgs
  Train patches: 6146
  Val patches:   1260
  Test patches:  1248
Dataset creation took 56.7s
Train: 6146 patches -> 769 batches


In [9]:
history_s3 = run_training_stage(
    model=model,
    train_loader=train_loader_s3,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    stage_name='stage3_unet_corrected',
    num_epochs=stage3_epochs,
    checkpoint_dir=CORRECTED_CKPT_DIR,
    save_every_epoch=True,
)

print(f"\nBranch 1 done. Best checkpoint: {os.path.join(CORRECTED_CKPT_DIR, 'stage3_unet_corrected_best.pth')}")



  Stage: stage3_unet_corrected
  Epochs: 20, LR: 0.0001, Patience: 7
  Train batches: 769, Val batches: 158
  [MPS Mem] Allocated: 164MB, Driver: 10452MB


  Epoch   1/20 │ Loss: 0.1768/0.1455 │ mIoU: 0.8445 │ Fines IoU: 0.4720 │ LR: 9.76e-05 │ 1199.3s
  [MPS Mem] Allocated: 374MB, Driver: 11502MB


  Epoch   2/20 │ Loss: 0.1717/0.1458 │ mIoU: 0.8512 │ Fines IoU: 0.5046 │ LR: 9.05e-05 │ 1184.3s
  [MPS Mem] Allocated: 374MB, Driver: 11494MB


  Epoch   3/20 │ Loss: 0.1681/0.1425 │ mIoU: 0.8495 │ Fines IoU: 0.4940 │ LR: 7.94e-05 │ 1170.4s
  [MPS Mem] Allocated: 375MB, Driver: 11494MB


  Epoch   4/20 │ Loss: 0.1650/0.1424 │ mIoU: 0.8518 │ Fines IoU: 0.5072 │ LR: 6.55e-05 │ 1171.9s
  [MPS Mem] Allocated: 374MB, Driver: 11494MB


  Epoch   5/20 │ Loss: 0.1622/0.1377 │ mIoU: 0.8519 │ Fines IoU: 0.4990 │ LR: 5.00e-05 │ 1174.6s
  [MPS Mem] Allocated: 375MB, Driver: 10490MB


... (11 outputs left out)

  Epoch  17/20 │ Loss: 0.1499/0.1320 │ mIoU: 0.8595 │ Fines IoU: 0.5331 │ LR: 7.27e-05 │ 1160.9s
  [MPS Mem] Allocated: 376MB, Driver: 10494MB


  Epoch  18/20 │ Loss: 0.1484/0.1343 │ mIoU: 0.8560 │ Fines IoU: 0.5174 │ LR: 6.55e-05 │ 1156.9s
  [MPS Mem] Allocated: 376MB, Driver: 10486MB


  Epoch  19/20 │ Loss: 0.1474/0.1262 │ mIoU: 0.8634 │ Fines IoU: 0.5390 │ LR: 5.78e-05 │ 1161.4s
  [MPS Mem] Allocated: 376MB, Driver: 10486MB


  Epoch  20/20 │ Loss: 0.1470/0.1306 │ mIoU: 0.8594 │ Fines IoU: 0.5221 │ LR: 5.00e-05 │ 1155.6s
  ✅ Restored best model from epoch 19 (val mIoU = 0.8634)

Branch 1 done. Best checkpoint: <project>/checkpoints/corrected_v1/stage3_unet_corrected_best.pth


## 2. Branch 2 — FinesDetector, 30 epochs

Identical recipe to `notebooks/28_twin_branch_fines.ipynb`: same architecture (10 dilated conv
layers, `base_channels=32`), same `CenterNetFocalLoss(alpha=2.0, beta=4.0)`, same
`AdamW(lr=3e-4, weight_decay=1e-4)`, same `CosineAnnealingWarmRestarts(T_0=30, T_mult=1,
eta_min=1e-6)`, same `patch_size=512, overlap=64, sigma=2.0, min_blob_px=3`, same
`EPOCHS=30, PATIENCE=10`. Reads the corrected `data_split.json` (already updated). Saves every
epoch, exactly as the original recipe already did.

In [10]:
def generate_centroid_heatmap(mask, sigma=2.0, min_blob_px=3):
    H, W = mask.shape
    heatmap = np.zeros((H, W), dtype=np.float32)
    fines_mask = (mask == 3).astype(np.uint8)
    labeled, n_blobs = label(fines_mask)
    if n_blobs == 0:
        return heatmap[np.newaxis, ...]
    slices = find_objects(labeled)
    radius = int(np.ceil(4.0 * sigma))
    for i, sl in enumerate(slices):
        if sl is None:
            continue
        blob_mask = (labeled[sl] == (i + 1))
        blob_size = int(blob_mask.sum())
        if blob_size < min_blob_px:
            continue
        y_idx, x_idx = np.nonzero(blob_mask)
        if len(y_idx) == 0:
            continue
        cy = float(y_idx.mean() + sl[0].start)
        cx = float(x_idx.mean() + sl[1].start)
        y0 = max(0, int(cy) - radius); y1 = min(H, int(cy) + radius + 1)
        x0 = max(0, int(cx) - radius); x1 = min(W, int(cx) + radius + 1)
        yy = np.arange(y0, y1, dtype=np.float32) - cy
        xx = np.arange(x0, x1, dtype=np.float32) - cx
        gaussian = np.exp(-(yy[:, None]**2 + xx[None, :]**2) / (2.0 * sigma * sigma))
        patch = heatmap[y0:y1, x0:x1]
        np.maximum(patch, gaussian, out=patch)
    return heatmap[np.newaxis, ...]


class FinesHeatmapDataset(SandPatchDataset):
    def __init__(self, image_indices, transform=None, patch_size=None,
                 overlap=None, sigma=2.0, min_blob_px=3):
        super().__init__(image_indices, transform, patch_size, overlap)
        self.sigma = sigma
        self.min_blob_px = min_blob_px

    def __getitem__(self, index):
        img = self.img_patches[index]
        mask = self.mask_patches[index]
        augmented = self.transform(image=img, mask=mask)
        img_t = augmented["image"]
        mask_aug = augmented["mask"]
        mask_np = mask_aug.numpy() if isinstance(mask_aug, torch.Tensor) else mask_aug
        heatmap = generate_centroid_heatmap(mask_np, self.sigma, self.min_blob_px)
        heatmap_t = torch.from_numpy(heatmap).float()
        return img_t, heatmap_t


class CenterNetFocalLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=4.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, pred_logits, target):
        pred = torch.sigmoid(pred_logits)
        pred = pred.clamp(1e-6, 1 - 1e-6)
        pos_mask = target.eq(1).float()
        neg_mask = target.lt(1).float()
        pos_loss = -((1 - pred) ** self.alpha) * torch.log(pred) * pos_mask
        neg_weight = ((1 - target) ** self.beta)
        neg_loss = -(pred ** self.alpha) * torch.log(1 - pred) * neg_weight * neg_mask
        num_pos = pos_mask.sum().clamp(min=1)
        return (pos_loss.sum() + neg_loss.sum()) / num_pos


In [11]:
# Free Branch 1's training data before starting Branch 2
del train_ds_s3, train_loader_s3
gc.collect()
if device.type == 'mps':
    torch.mps.synchronize()
    torch.mps.empty_cache()

with open(os.path.join(CFG.project_root, "data_split.json")) as f:
    split = json.load(f)

train_ds_b2 = FinesHeatmapDataset(
    split["train"], transform=get_train_augmentation(),
    patch_size=512, overlap=64, sigma=2.0, min_blob_px=3
)
val_ds_b2 = FinesHeatmapDataset(
    split["val"], transform=get_val_augmentation(),
    patch_size=512, overlap=64, sigma=2.0, min_blob_px=3
)

train_loader_b2 = DataLoader(train_ds_b2, batch_size=8, shuffle=True, num_workers=0, pin_memory=False)
val_loader_b2 = DataLoader(val_ds_b2, batch_size=8, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train: {len(train_ds_b2)} patches ({len(train_loader_b2)} batches)")
print(f"Val:   {len(val_ds_b2)} patches ({len(val_loader_b2)} batches)")


Train: 6146 patches (769 batches)
Val:   1260 patches (158 batches)


In [12]:
model_b2 = FinesDetector(base_channels=32).to(device)
criterion_b2 = CenterNetFocalLoss(alpha=2.0, beta=4.0).to(device)
optimizer_b2 = torch.optim.AdamW(model_b2.parameters(), lr=3e-4, weight_decay=1e-4)

EPOCHS_B2 = 30
PATIENCE_B2 = 10

scheduler_b2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_b2, T_0=EPOCHS_B2, T_mult=1, eta_min=1e-6
)

best_val_loss_b2 = float("inf")
epochs_no_improve_b2 = 0


def train_epoch_b2(epoch):
    model_b2.train()
    total_loss = 0.0
    if device.type == "mps":
        torch.mps.empty_cache()
    pbar = tqdm(train_loader_b2, desc=f"Train E{epoch+1}")
    for i, (imgs, heatmaps) in enumerate(pbar):
        imgs = imgs.to(device); heatmaps = heatmaps.to(device)
        pred = model_b2(imgs)
        loss = criterion_b2(pred, heatmaps)
        optimizer_b2.zero_grad()
        loss.backward()
        optimizer_b2.step()
        total_loss += loss.item()
        if device.type == "mps" and i % 100 == 0:
            torch.mps.empty_cache()
        if i % 20 == 0:
            pbar.set_postfix(loss=f"{total_loss/(i+1):.4f}")
    return total_loss / len(train_loader_b2)


def val_epoch_b2(epoch):
    model_b2.eval()
    total_loss = 0.0
    if device.type == "mps":
        torch.mps.empty_cache()
    pbar = tqdm(val_loader_b2, desc=f"Val   E{epoch+1}")
    with torch.no_grad():
        for imgs, heatmaps in pbar:
            imgs = imgs.to(device); heatmaps = heatmaps.to(device)
            pred = model_b2(imgs)
            loss = criterion_b2(pred, heatmaps)
            total_loss += loss.item()
    return total_loss / len(val_loader_b2)


In [13]:
print(f"{'='*70}")
print(f"  Branch 2 Training: FinesDetector (corrected data)")
print(f"  Epochs: {EPOCHS_B2}, Patience: {PATIENCE_B2}")
print(f"{'='*70}")

history_b2 = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS_B2):
    t0 = time.time()
    tr_loss = train_epoch_b2(epoch)
    vl_loss = val_epoch_b2(epoch)
    scheduler_b2.step()
    elapsed = time.time() - t0
    history_b2["train_loss"].append(tr_loss)
    history_b2["val_loss"].append(vl_loss)
    lr = optimizer_b2.param_groups[0]["lr"]
    print(f"Epoch {epoch+1:2d}/{EPOCHS_B2} | Loss: {tr_loss:.4f}/{vl_loss:.4f} | LR: {lr:.2e} | {elapsed:.1f}s")

    # Save every epoch (weights only)
    ckpt_path = os.path.join(CORRECTED_CKPT_DIR, f"fines_detector_corrected_e{epoch+1:02d}.pth")
    torch.save(model_b2.state_dict(), ckpt_path)

    if vl_loss < best_val_loss_b2:
        best_val_loss_b2 = vl_loss
        epochs_no_improve_b2 = 0
        best_path = os.path.join(CORRECTED_CKPT_DIR, "fines_detector_corrected_best.pth")
        torch.save(model_b2.state_dict(), best_path)
        print(f"  [+] New best (val_loss: {best_val_loss_b2:.4f})")
    else:
        epochs_no_improve_b2 += 1

    if epochs_no_improve_b2 >= PATIENCE_B2:
        print(f"\nEarly stopping after {epoch+1} epochs.")
        break

with open(os.path.join(CORRECTED_CKPT_DIR, "fines_detector_corrected_history.json"), "w") as f:
    json.dump(history_b2, f, indent=2)

print(f"\nBest val loss: {best_val_loss_b2:.4f}")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_b2["train_loss"], label="Train")
ax.plot(history_b2["val_loss"], label="Val")
ax.set_xlabel("Epoch"); ax.set_ylabel("CenterNet Focal Loss")
ax.set_title("Branch 2 Training (corrected data)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


  Branch 2 Training: FinesDetector (corrected data)
  Epochs: 30, Patience: 10


Val   E1: 100%|██████████| 158/158 [01:17<00:00,  2.03it/s]


Epoch  1/30 | Loss: 1517.0508/603.9318 | LR: 2.99e-04 | 1556.2s
  [+] New best (val_loss: 603.9318)


Val   E2: 100%|██████████| 158/158 [01:18<00:00,  2.00it/s]


Epoch  2/30 | Loss: 129.7226/170.9062 | LR: 2.97e-04 | 1547.1s
  [+] New best (val_loss: 170.9062)


Val   E3: 100%|██████████| 158/158 [01:18<00:00,  2.02it/s]


... (51 outputs left out)

Val   E29: 100%|██████████| 158/158 [01:17<00:00,  2.03it/s]


Epoch 29/30 | Loss: 1.6211/1.4717 | LR: 1.82e-06 | 1551.9s
  [+] New best (val_loss: 1.4717)


Val   E30: 100%|██████████| 158/158 [01:18<00:00,  2.02it/s]


Epoch 30/30 | Loss: 1.6104/1.4937 | LR: 3.00e-04 | 1553.6s

Best val loss: 1.4717


## 3. Load both branches, official evaluation on the corrected 16-image test set

Primary evaluation uses the exact confirmed fusion config — `fusion_scale=8.0,
fusion_margin=2.0`, **no TTA** — matching the original baseline's methodology exactly. This is
the number that becomes the new official baseline.

In [14]:
# Free Branch 2 training data
del train_ds_b2, train_loader_b2, val_ds_b2, val_loader_b2
gc.collect()
if device.type == 'mps':
    torch.mps.synchronize()
    torch.mps.empty_cache()

# Reload Branch 1 best-of-run (already in memory as `model`, but reload from disk to be certain
# we're evaluating exactly what was checkpointed, not a stale in-memory state)
model_b1 = build_unet()
b1_ckpt = torch.load(os.path.join(CORRECTED_CKPT_DIR, "stage3_unet_corrected_best.pth"), map_location="cpu")
b1_state = b1_ckpt["model_state_dict"] if isinstance(b1_ckpt, dict) and "model_state_dict" in b1_ckpt else b1_ckpt
model_b1.load_state_dict(b1_state)
model_b1 = model_b1.to(device)
model_b1.eval()

model_b2_eval = FinesDetector(base_channels=32)
model_b2_eval.load_state_dict(torch.load(os.path.join(CORRECTED_CKPT_DIR, "fines_detector_corrected_best.pth"), map_location="cpu"))
model_b2_eval = model_b2_eval.to(device)
model_b2_eval.eval()

print("Both branches loaded from corrected_v1 checkpoints.")


def fused_predict_fn(model, image_path, device, use_tta=False, fusion_margin=2.0):
    return predict_full_image(
        model=model_b1,
        image_path=image_path,
        device=device,
        model_b2=model_b2_eval,
        fusion_scale=8.0,
        fusion_margin=fusion_margin,
        use_tta=use_tta,
    )


Both branches loaded from corrected_v1 checkpoints.


In [15]:
with open(os.path.join(CFG.project_root, "data_split.json")) as f:
    split = json.load(f)
test_indices = split["test"]
val_indices = split["val"]
print(f"Evaluating on the corrected test set: {len(test_indices)} images -> {test_indices}")

metrics_primary, gt_blobs_total, gt_blobs_detected = run_official_evaluation(
    model=model_b1,
    device=device,
    test_indices=test_indices,
    experiment_name="Corrected_GatedFusion_NoTTA",
    use_tta=False,
    custom_predict_fn=lambda model, path, dev: fused_predict_fn(model, path, dev, use_tta=False),
)

s_primary = metrics_primary.summary()
total_real = sum(v for k, v in gt_blobs_total.items() if 'noise' not in k)
total_real_det = sum(gt_blobs_detected[k] for k in gt_blobs_total if 'noise' not in k)
primary_fines_iou = s_primary['Fines_iou']
primary_det_rate = (total_real_det / total_real * 100) if total_real > 0 else 0.0

print(f"\n>>> PRIMARY (new official) result — Fines IoU: {primary_fines_iou:.4f}, "
      f"Blob Detection (3px+): {primary_det_rate:.1f}%")


Evaluating on the corrected test set: 16 images -> [1, 5, 11, 12, 19, 27, 32, 43, 46, 57, 70, 74, 89, 91, 97, 113]

  OFFICIAL EVALUATION: Corrected_GatedFusion_NoTTA
  Test Set: 16 images

  Processed Image_001 in 8.9s
  Processed Image_005 in 9.4s
  Processed Image_011 in 8.6s
  Processed Image_012 in 8.6s
  Processed Image_019 in 8.5s
  Processed Image_027 in 8.5s
  Processed Image_032 in 8.6s
  Processed Image_043 in 8.6s
  Processed Image_046 in 8.6s
  Processed Image_057 in 8.6s
  Processed Image_070 in 7.6s
  Processed Image_074 in 8.5s
  Processed Image_089 in 7.5s
  Processed Image_091 in 7.5s
  Processed Image_097 in 7.5s
  Processed Image_113 in 8.6s

Evaluation completed in 144.4s

Saved Confusion Matrix to: <project>/outputs/20260830_061205_Corrected_GatedFusion_NoTTA_cm.png



  PER-CLASS ACCURACY TABLE: Corrected_GatedFusion_NoTTA
  Overall Pixel Accuracy : 0.9826
  Mean IoU (mIoU)        : 0.8370
───────────────────────────────────────────────────────────────────────────
  Class           IoU  Precision   Recall  F1/Dice
  ───────────────────────────────────────────────────────
  Grains       0.9751     0.9912   0.9836   0.9874
  Void         0.9536     0.9714   0.9811   0.9762
  Water        0.9572     0.9728   0.9835   0.9781
  Fines        0.4620     0.5727   0.7050   0.6320

  BLOB DETECTION BREAKDOWN: Corrected_GatedFusion_NoTTA
  Size Bucket        Detected /    Total      Rate
  ───────────────────────────────────────────────────────
  ⚪ 1-2px (noise)          9168 /    36624     25.0%
  ⚠️ 3px                    2930 /     4849     60.4%
  ⚠️ 4px                    4368 /     5446     80.2%
  ⚠️ 5-6px                  6171 /     7270     84.9%
  ⚠️ 7-10px                 9496 /    10705     88.7%
  ⚠️ 11-26px                7202 /     7626     94.

## 4. Finishing steps (supplementary, matching how the original baseline was finalized)

Two variants, evaluated the same way the original confirmed config was explored:
1. **TTA on**, same fusion config — direct on/off comparison.
2. **`fusion_margin` sweep**, selected on the *val* set and reported on *test* — the same
   val-select/test-report calibration pattern used originally to settle on `fusion_margin=2.0`.

Neither of these replaces the primary no-TTA result above as the official number unless one of
them is clearly better *and* you decide to adopt it — the original confirmed baseline itself used
no TTA, so the primary result is the direct like-for-like comparison point.

In [16]:
# 4a. TTA variant
metrics_tta, gt_blobs_total_tta, gt_blobs_detected_tta = run_official_evaluation(
    model=model_b1,
    device=device,
    test_indices=test_indices,
    experiment_name="Corrected_GatedFusion_TTA",
    use_tta=True,
    custom_predict_fn=lambda model, path, dev: fused_predict_fn(model, path, dev, use_tta=True),
)
s_tta = metrics_tta.summary()
total_real_tta = sum(v for k, v in gt_blobs_total_tta.items() if 'noise' not in k)
total_real_det_tta = sum(gt_blobs_detected_tta[k] for k in gt_blobs_total_tta if 'noise' not in k)
tta_fines_iou = s_tta['Fines_iou']
tta_det_rate = (total_real_det_tta / total_real_tta * 100) if total_real_tta > 0 else 0.0
print(f"\nTTA variant — Fines IoU: {tta_fines_iou:.4f}, Blob Detection: {tta_det_rate:.1f}%")



  OFFICIAL EVALUATION: Corrected_GatedFusion_TTA
  Test Set: 16 images

  Processed Image_001 in 18.7s
  Processed Image_005 in 20.3s
  Processed Image_011 in 18.5s
  Processed Image_012 in 18.5s
  Processed Image_019 in 18.5s
  Processed Image_027 in 18.5s
  Processed Image_032 in 18.5s
  Processed Image_043 in 18.6s
  Processed Image_046 in 18.5s
  Processed Image_057 in 18.6s
  Processed Image_070 in 16.3s
  Processed Image_074 in 18.6s
  Processed Image_089 in 16.3s
  Processed Image_091 in 16.3s
  Processed Image_097 in 16.3s
  Processed Image_113 in 18.6s

Evaluation completed in 299.9s

Saved Confusion Matrix to: <project>/outputs/20260830_061705_Corrected_GatedFusion_TTA_cm.png



  PER-CLASS ACCURACY TABLE: Corrected_GatedFusion_TTA
  Overall Pixel Accuracy : 0.9831
  Mean IoU (mIoU)        : 0.8396
───────────────────────────────────────────────────────────────────────────
  Class           IoU  Precision   Recall  F1/Dice
  ───────────────────────────────────────────────────────
  Grains       0.9758     0.9915   0.9840   0.9878
  Void         0.9549     0.9722   0.9817   0.9769
  Water        0.9582     0.9733   0.9841   0.9787
  Fines        0.4697     0.5838   0.7062   0.6392

  BLOB DETECTION BREAKDOWN: Corrected_GatedFusion_TTA
  Size Bucket        Detected /    Total      Rate
  ───────────────────────────────────────────────────────
  ⚪ 1-2px (noise)          8852 /    36624     24.2%
  ⚠️ 3px                    2919 /     4849     60.2%
  ⚠️ 4px                    4356 /     5446     80.0%
  ⚠️ 5-6px                  6149 /     7270     84.6%
  ⚠️ 7-10px                 9495 /    10705     88.7%
  ⚠️ 11-26px                7201 /     7626     94.4%
 

In [17]:
# 4b. fusion_margin sweep — select on val, report on test
def evaluate_fusion_margin(image_indices, margin):
    metrics = SegmentationMetrics()
    gt_tot, gt_det = 0, 0
    for idx in image_indices:
        raw_path = CFG.raw_path(idx)
        gt_path = CFG.seg_path(idx)
        gt_img = np.array(Image.open(gt_path))
        rgb_out, pred_map, elapsed = fused_predict_fn(model_b1, raw_path, device, use_tta=False, fusion_margin=margin)
        min_h = min(pred_map.shape[0], gt_img.shape[0])
        min_w = min(pred_map.shape[1], gt_img.shape[1])
        pred_map_c = pred_map[:min_h, :min_w]
        gt_map = rgb_to_class_index(gt_img[:min_h, :min_w])
        metrics.update(pred_map_c, gt_map)
        gt_fines = (gt_map == 3).astype(np.uint8)
        pred_fines = (pred_map_c == 3).astype(np.uint8)
        labeled_gt, _ = label(gt_fines)
        for sl in find_objects(labeled_gt):
            if sl is None:
                continue
            blob_mask = (labeled_gt[sl] > 0)
            size = int(blob_mask.sum())
            if size < 3:
                continue  # exclude 1-2px noise, matching the official protocol
            gt_tot += 1
            if np.sum(blob_mask & (pred_fines[sl] == 1)) > 0:
                gt_det += 1
    s = metrics.summary()
    det_rate = (gt_det / gt_tot * 100) if gt_tot > 0 else 0.0
    return s['Fines_iou'], det_rate


margin_grid = [1.0, 1.5, 2.0, 2.5, 3.0]
print("Sweeping fusion_margin on VAL set...")
val_results = {}
for m in margin_grid:
    iou, det = evaluate_fusion_margin(val_indices, m)
    val_results[m] = (iou, det)
    print(f"  margin={m:.1f} -> Val Fines IoU={iou:.4f}, Val Det={det:.1f}%")

best_margin = max(val_results, key=lambda m: val_results[m][1])
print(f"\nBest margin by val detection rate: {best_margin}")

print(f"\nEvaluating margin={best_margin} on TEST...")
sweep_test_iou, sweep_test_det = evaluate_fusion_margin(test_indices, best_margin)
print(f"Sweep result (margin={best_margin}) on test — Fines IoU: {sweep_test_iou:.4f}, "
      f"Blob Detection: {sweep_test_det:.1f}%")


Sweeping fusion_margin on VAL set...
  margin=1.0 -> Val Fines IoU=0.5304, Val Det=92.0%
  margin=1.5 -> Val Fines IoU=0.5302, Val Det=92.0%
  margin=2.0 -> Val Fines IoU=0.5302, Val Det=92.0%
  margin=2.5 -> Val Fines IoU=0.5302, Val Det=92.0%
  margin=3.0 -> Val Fines IoU=0.5302, Val Det=92.0%

Best margin by val detection rate: 2.0

Evaluating margin=2.0 on TEST...
Sweep result (margin=2.0) on test — Fines IoU: 0.4620, Blob Detection: 84.1%


## 5. Final report — new official baseline vs. the old one (not directly comparable)

In [18]:
OLD_FINES_IOU = 0.4179
OLD_DETECTION = 76.1
OLD_TEST_N = 18

NEW_TEST_N = len(test_indices)

print("="*78)
print("  OFFICIAL RESULT COMPARISON — read the caveat before drawing conclusions")
print("="*78)
print(f"{'':28s}{'OLD baseline':>18s}{'NEW baseline':>18s}")
print(f"{'Test set size':28s}{OLD_TEST_N:>18d}{NEW_TEST_N:>18d}")
print(f"{'Fines IoU':28s}{OLD_FINES_IOU:>18.4f}{primary_fines_iou:>18.4f}")
print(f"{'Blob Detection (3px+)':28s}{OLD_DETECTION:>17.1f}%{primary_det_rate:>17.1f}%")
print("="*78)
print("""
CAVEAT: these two numbers are NOT a clean apples-to-apples comparison. The old baseline
(76.1% / 0.4179) was measured on the original 18-image test set, which — as this investigation
found — secretly contained one exact duplicate (Image_089/090) and one image with a confirmed,
unfixable label-registration problem (Image_119). The new baseline is measured on a corrected,
de-duplicated 16-image test set with that problem removed.

Any difference between the two numbers reflects BOTH:
  1. the corrected training data (the actual thing being tested here), and
  2. a fairer test set (two fewer, cleaner images) — which changes what's being measured, not
     just how well the model does on it.

This result should be reported and used as the new official baseline going forward, not framed
as a direct 'improvement' or 'regression' relative to 76.1% / 0.4179.
""")

print("\nSupplementary (not official):")
print(f"  TTA variant:            Fines IoU={tta_fines_iou:.4f}, Detection={tta_det_rate:.1f}%")
print(f"  fusion_margin={best_margin} (val-selected): Fines IoU={sweep_test_iou:.4f}, Detection={sweep_test_det:.1f}%")


  OFFICIAL RESULT COMPARISON — read the caveat before drawing conclusions
                                  OLD baseline      NEW baseline
Test set size                               18                16
Fines IoU                               0.4179            0.4620
Blob Detection (3px+)                    76.1%             84.1%

CAVEAT: these two numbers are NOT a clean apples-to-apples comparison. The old baseline
(76.1% / 0.4179) was measured on the original 18-image test set, which — as this investigation
found — secretly contained one exact duplicate (Image_089/090) and one image with a confirmed,
unfixable label-registration problem (Image_119). The new baseline is measured on a corrected,
de-duplicated 16-image test set with that problem removed.

Any difference between the two numbers reflects BOTH:
  1. the corrected training data (the actual thing being tested here), and
  2. a fairer test set (two fewer, cleaner images) — which changes what's being measured, not
     just